In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [ ]:
!pip install -q torchtext

In [ ]:
!pip install -q datasets
from datasets import load_dataset
# Load the dataset
data = load_dataset("bentrevett/multi30k")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.tensorboard import SummaryWriter

In [ ]:
import spacy

In [ ]:
!python -m spacy download en_core_web_sm
!python -m spacy download de_core_news_sm

In [ ]:
en_nlp = spacy.load('en_core_web_sm')
de_nlp = spacy.load('de_core_news_sm')

In [ ]:
def tokenize_en(text):
  return [token.text for token in en_nlp.tokenizer(text)]
def tokenize_de(text):
    return [token.text for token in de_nlp.tokenizer(text)]
sample_text = data['train'][0]['en']
print(f"Original: {sample_text}")
print(f"Tokenized: {tokenize_en(sample_text)}")

Original: Two young, White males are outside near many bushes.
Tokenized: ['Two', 'young', ',', 'White', 'males', 'are', 'outside', 'near', 'many', 'bushes', '.']


In [ ]:
def tokenize_dataset(example):
    example['en'] = tokenize_en(example['en'])
    example['de'] = tokenize_de(example['de'])
    return example
tokenized_dataset = data.map(tokenize_dataset)
print("Old format:", data['train'][0]['en'])
print("New format:", tokenized_dataset['train'][0]['en'])

Map:   0%|          | 0/1014 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Old format: Two young, White males are outside near many bushes.
New format: ['Two', 'young', ',', 'White', 'males', 'are', 'outside', 'near', 'many', 'bushes', '.']


In [ ]:
from collections import Counter
en_counts = Counter()
de_counts = Counter()
for example in tokenized_dataset['train']:
    en_counts.update(example['en'])
    de_counts.update(example['de'])
en_vocab = {word: i+4 for i, (word, count) in enumerate(en_counts.items()) if count > 1}
de_vocab = {word: i+4 for i, (word, count) in enumerate(de_counts.items()) if count > 1}
special_tokens = {"<unk>": 0, "<pad>": 1, "<sos>": 2, "<eos>": 3}
en_vocab.update(special_tokens)
de_vocab.update(special_tokens)
print(f"Value of 'young': {en_vocab.get('young', 'Not Found')}")


Value of 'young': 5


In [ ]:
def numericalize(example):
    example['en'] = [en_vocab.get(word, 0) for word in example['en']]
    example['de'] = [de_vocab.get(word, 0) for word in example['de']]
    return example

numericalized_dataset = tokenized_dataset.map(numericalize)

Map:   0%|          | 0/1014 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
def collate_batch(batch):
    src_list = []
    trg_list = []
    for item in batch:
        src_tensor = torch.tensor([2] + item['en'] + [3])
        trg_tensor = torch.tensor([2] + item['de'] + [3])

        src_list.append(src_tensor)
        trg_list.append(trg_tensor)
    src_padded = pad_sequence(src_list, padding_value=1)
    trg_padded = pad_sequence(trg_list, padding_value=1)

    return src_padded, trg_padded


train_loader = DataLoader(numericalized_dataset['train'], batch_size=64, collate_fn=collate_batch)

In [ ]:
import torch
import torch.nn as nn
import random

class Encoder(nn.Module):
  def __init__(self, input_size, embedding_size, hidden_size, num_layers, p):
    super(Encoder, self).__init__()
    self.hidden_size = hidden_size
    self.num_layers = num_layers
    self.dropout = nn.Dropout(p)
    self.embedding = nn.Embedding(input_size, embedding_size)
    self.rnn = nn.LSTM(embedding_size, hidden_size, num_layers, dropout=p)

  def forward(self, x):
    embedding = self.dropout(self.embedding(x))
    outputs, (hidden, cell) = self.rnn(embedding)
    return hidden, cell

class Decoder(nn.Module):
  def __init__(self, input_size, embedding_size, hidden_size, output_size, num_layers, p):
    super(Decoder, self).__init__()
    self.hidden_size = hidden_size
    self.num_layers = num_layers
    self.dropout = nn.Dropout(p)
    self.embedding = nn.Embedding(input_size, embedding_size)
    self.rnn = nn.LSTM(embedding_size, hidden_size, num_layers, dropout=p)
    self.fc = nn.Linear(hidden_size, output_size)


  def forward(self, x, hidden, cell):

    x = x.unsqueeze(0)
    embedding = self.dropout(self.embedding(x))
    outputs, (hidden, cell) = self.rnn(embedding, (hidden, cell))

    predictions = self.fc(outputs)
    predictions = predictions.squeeze(0)
    return predictions, hidden, cell

class Seq2Seq(nn.Module):
  def __init__(self, encoder, decoder):
    super(Seq2Seq, self).__init__()
    self.encoder = encoder
    self.decoder = decoder


  def forward(self, source, target, teacher_force_ratio=0.5):
    batch_size = source.shape[1]
    target_len = target.shape[0]


    target_vocab_size = self.decoder.fc.out_features
    outputs = torch.zeros(target_len, batch_size, target_vocab_size).to(source.device)

    hidden, cell = self.encoder(source)
    x = target[0]

    for t in range(1, target_len):
      output, hidden, cell = self.decoder(x, hidden, cell)
      outputs[t] = output
      best_guess = output.argmax(1)
      x = target[t] if random.random() < teacher_force_ratio else best_guess

    return outputs

In [ ]:

input_size_en = max(en_vocab.values()) + 1
input_size_de = max(de_vocab.values()) + 1

print(input_size_en, input_size_de)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
embedding_size = 256
hidden_size = 512
num_layers = 2
dropout = 0.5
learning_rate = 0.001
num_epochs = 25
step = 0

encoder = Encoder(input_size_en, embedding_size, hidden_size, num_layers, dropout)
decoder = Decoder(input_size_de, embedding_size, hidden_size, input_size_de, num_layers, dropout)
model = Seq2Seq(encoder, decoder).to(device)

writer = SummaryWriter()

In [ ]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=learning_rate)
pad_idx = 1
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

for epoch in range(num_epochs):
    print(f"--- Starting Epoch {epoch+1} / {num_epochs} ---")
    model.train()


    epoch_loss = 0

    for batch_idx, (src_data, trg_data) in enumerate(train_loader):
        src_data = src_data.to(device)
        trg_data = trg_data.to(device)

        output = model(src_data, trg_data)
        output = output[1:].view(-1, output.shape[-1])
        trg_data = trg_data[1:].view(-1)

        optimizer.zero_grad()
        loss = criterion(output, trg_data)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)
        optimizer.step()

        epoch_loss += loss.item()
        writer.add_scalar("Training Loss", loss.item(), global_step=step)
        step += 1


    avg_loss = epoch_loss / len(train_loader)


    print(f"Epoch {epoch+1} completed. Average Loss: {avg_loss:.4f}")

In [ ]:
def translate(sentence, en_vocab, de_vocab, model, device, max_len=50):
    model.eval()
    tokens = [en_vocab.get(word, 0) for word in tokenize_en(sentence)]
    src = torch.tensor([2] + tokens + [3]).unsqueeze(1).to(device)

    with torch.no_grad():
        hidden, cell = model.encoder(src)

    de_idx_to_word = {v: k for k, v in de_vocab.items()}
    x = torch.tensor([2]).to(device)
    translated = []

    for _ in range(max_len):
        with torch.no_grad():
            output, hidden, cell = model.decoder(x, hidden, cell)
        pred = output.argmax(1)
        if pred.item() == 3:  # <eos>
            break
        translated.append(de_idx_to_word.get(pred.item(), '<unk>'))
        x = pred

    return " ".join(translated)


print(translate("A dog is running in the park", en_vocab, de_vocab, model, device))

In [ ]:
from nltk.translate.bleu_score import corpus_bleu
import nltk
nltk.download('punkt')
from nltk.translate.bleu_score import corpus_bleu
def evaluate_bleu(dataset, en_vocab, de_vocab, model, device, n=500):
    references = []
    hypotheses = []
    de_idx_to_word = {v: k for k, v in de_vocab.items()}
    en_idx_to_word = {v: k for k, v in en_vocab.items()}

    test_data = dataset['test']

    for i in range(min(n, len(test_data))):
        example = test_data[i]


        src_words = [en_idx_to_word.get(idx, '<unk>')
                     for idx in example['en']
                     if idx not in [0, 1, 2, 3]]
        src_sentence = " ".join(src_words)


        translated = translate(src_sentence, en_vocab, de_vocab, model, device)


        ref = [de_idx_to_word.get(idx, '')
               for idx in example['de']
               if idx not in [0, 1, 2, 3]]
        hyp = translated.split()

        references.append([ref])
        hypotheses.append(hyp)

    score = corpus_bleu(references, hypotheses)
    print(f"BLEU Score: {score*100:.2f}")
    return score

evaluate_bleu(numericalized_dataset, en_vocab, de_vocab, model, device)

In [ ]:
class Attention(nn.Module):
    def __init__(self, hidden_size):
        super(Attention, self).__init__()
        self.W1 = nn.Linear(hidden_size, hidden_size)
        self.W2 = nn.Linear(hidden_size, hidden_size)
        self.V  = nn.Linear(hidden_size, 1)

    def forward(self, hidden, encoder_outputs):


        hidden = hidden[-1].unsqueeze(0)
        # [1, batch, hidden_size]

        energy = torch.tanh(self.W1(hidden) + self.W2(encoder_outputs))
        # [src_len, batch, hidden_size]

        attention = self.V(energy).squeeze(2)
        # [src_len, batch]

        return torch.softmax(attention, dim=0)
        # [src_len, batch] probability over source words

In [ ]:
class AttentionEncoder(nn.Module):
    def __init__(self, input_size, embedding_size, hidden_size, num_layers, p):
        super(AttentionEncoder, self).__init__()
        self.dropout = nn.Dropout(p)
        self.embedding = nn.Embedding(input_size, embedding_size)
        self.rnn = nn.LSTM(embedding_size, hidden_size, num_layers, dropout=p)

    def forward(self, x):
        embedding = self.dropout(self.embedding(x))
        outputs, (hidden, cell) = self.rnn(embedding)
        return outputs, hidden, cell

In [ ]:
class AttentionDecoder(nn.Module):
    def __init__(self, input_size, embedding_size, hidden_size, output_size, num_layers, p, attention):
        super(AttentionDecoder, self).__init__()
        self.dropout = nn.Dropout(p)
        self.embedding = nn.Embedding(input_size, embedding_size)
        self.attention = attention
        self.rnn = nn.LSTM(
            embedding_size + hidden_size,
            hidden_size,
            num_layers,
            dropout=p
        )
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden, cell, encoder_outputs):
        x = x.unsqueeze(0)
        embedding = self.dropout(self.embedding(x))

        attention_weights = self.attention(hidden, encoder_outputs)
        attention_weights = attention_weights.unsqueeze(2)

        encoder_outputs_permuted = encoder_outputs.permute(1, 0, 2)
        attention_weights_permuted = attention_weights.permute(1, 2, 0)

        context = torch.bmm(attention_weights_permuted, encoder_outputs_permuted)
        context = context.permute(1, 0, 2)

        rnn_input = torch.cat([embedding, context], dim=2)

        outputs, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))
        predictions = self.fc(outputs.squeeze(0))
        return predictions, hidden, cell

In [ ]:
class AttentionSeq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super(AttentionSeq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, source, target, teacher_force_ratio=0.5):
        batch_size = source.shape[1]
        target_len = target.shape[0]
        target_vocab_size = self.decoder.fc.out_features
        outputs = torch.zeros(target_len, batch_size, target_vocab_size).to(source.device)

        encoder_outputs, hidden, cell = self.encoder(source)  # ← now gets encoder_outputs too
        x = target[0]

        for t in range(1, target_len):
            output, hidden, cell = self.decoder(x, hidden, cell, encoder_outputs)  # ← passes encoder_outputs
            outputs[t] = output
            best_guess = output.argmax(1)
            x = target[t] if random.random() < teacher_force_ratio else best_guess

        return outputs

In [ ]:
input_size_en = max(en_vocab.values()) + 1
input_size_de = max(de_vocab.values()) + 1

print(input_size_en, input_size_de)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
embedding_size = 256
hidden_size = 512
num_layers = 2
dropout = 0.5
learning_rate = 0.001
num_epochs = 5
step = 0

In [ ]:

attention = Attention(hidden_size)
attention_encoder = AttentionEncoder(input_size_en, embedding_size, hidden_size, num_layers, dropout)
attention_decoder = AttentionDecoder(input_size_de, embedding_size, hidden_size, input_size_de, num_layers, dropout, attention)
attention_model = AttentionSeq2Seq(attention_encoder, attention_decoder).to(device)

attention_optimizer = optim.Adam(attention_model.parameters(), lr=learning_rate)

In [ ]:
import random
criterion = nn.CrossEntropyLoss(ignore_index=1)
attention_step = 0
attention_writer = SummaryWriter("runs/attention")

for epoch in range(num_epochs):
    print(f"--- Starting Epoch {epoch+1} / {num_epochs} ---")
    attention_model.train()
    epoch_loss = 0

    for batch_idx, (src_data, trg_data) in enumerate(train_loader):
        src_data = src_data.to(device)
        trg_data = trg_data.to(device)

        output = attention_model(src_data, trg_data)
        output = output[1:].view(-1, output.shape[-1])
        trg_data = trg_data[1:].view(-1)

        attention_optimizer.zero_grad()
        loss = criterion(output, trg_data)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(attention_model.parameters(), max_norm=1)
        attention_optimizer.step()

        epoch_loss += loss.item()
        attention_writer.add_scalar("Attention Training Loss", loss.item(), global_step=attention_step)
        attention_step += 1

    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1} completed. Average Loss: {avg_loss:.4f}")

In [ ]:
def translate(sentence, en_vocab, de_vocab, model, device, max_len=50):
    model.eval()
    tokens = [en_vocab.get(word, 0) for word in tokenize_en(sentence)]
    src = torch.tensor([2] + tokens + [3]).unsqueeze(1).to(device)

    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(src)

    de_idx_to_word = {v: k for k, v in de_vocab.items()}
    x = torch.tensor([2]).to(device)
    translated = []

    for _ in range(max_len):
        with torch.no_grad():
            output, hidden, cell = model.decoder(x, hidden, cell, encoder_outputs)
        pred = output.argmax(1)
        if pred.item() == 3:
            break
        translated.append(de_idx_to_word.get(pred.item(), '<unk>'))
        x = pred

    return " ".join(translated)

In [ ]:
from nltk.translate.bleu_score import corpus_bleu
import nltk
nltk.download('punkt')

def evaluate_bleu(dataset, en_vocab, de_vocab, model, device, n=500):
    references = []
    hypotheses = []
    de_idx_to_word = {v: k for k, v in de_vocab.items()}
    en_idx_to_word = {v: k for k, v in en_vocab.items()}

    test_data = dataset['test']

    for i in range(min(n, len(test_data))):
        example = test_data[i]

        src_words = [en_idx_to_word.get(idx, '<unk>')
                     for idx in example['en']
                     if idx not in [0, 1, 2, 3]]
        src_sentence = " ".join(src_words)

        translated = translate(src_sentence, en_vocab, de_vocab, model, device)

        ref = [de_idx_to_word.get(idx, '')
               for idx in example['de']
               if idx not in [0, 1, 2, 3]]
        hyp = translated.split()

        references.append([ref])
        hypotheses.append(hyp)

    score = corpus_bleu(references, hypotheses)
    print(f"BLEU Score: {score*100:.2f}")
    return score


print("=== Attention Seq2Seq BLEU ===")
evaluate_bleu(numericalized_dataset, en_vocab, de_vocab, attention_model, device)


print("\n=== Translation Test ===")
test_sentences = [
    "A dog is running in the park",
    "A man is playing football",
    "A woman is reading a book",
]

for sentence in test_sentences:
    translation = translate(sentence, en_vocab, de_vocab, attention_model, device)
    print(f"EN: {sentence}")
    print(f"DE: {translation}")
    print()